In [0]:
# Week 05 - Inspect Bronze schemas

print("=== METERS BRONZE SCHEMA ===")
spark.table("workspace.gridpulse_bronze.meters_bronze").printSchema()

print("\n=== BUILDINGS BRONZE SCHEMA ===")
spark.table("workspace.gridpulse_bronze.buildings_bronze").printSchema()

print("\n=== CONSUMPTION BRONZE SCHEMA ===")
spark.table("workspace.gridpulse_bronze.consumption_bronze").printSchema()

=== METERS BRONZE SCHEMA ===
root
 |-- source_record_id: string (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- meter_serial_no: string (nullable = true)
 |-- building_id: string (nullable = true)
 |-- tariff_plan_id: string (nullable = true)
 |-- meter_type: string (nullable = true)
 |-- capacity_kw: double (nullable = true)
 |-- meter_status: string (nullable = true)
 |-- voltage_class: string (nullable = true)
 |-- installed_date: date (nullable = true)
 |-- effective_from: timestamp (nullable = true)
 |-- effective_to: timestamp (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)


=== BUILDINGS BRONZE SCHEMA ===
root
 |-- active_flag: boolean (nullable = true)
 |-- building_id: string (nullable = true)
 |-- building_name: string (nullable = true)
 |-- building_type: string (nullable = true)
 |-- campus_zone: string (nullable = true)
 |-- commissioning_date: string (nullable = true)
 |-- criticality_ban

In [0]:
print("=== BUILDINGS BRONZE SCHEMA ===")
spark.table("workspace.gridpulse_bronze.buildings_bronze").printSchema()

print("\n=== CONSUMPTION BRONZE SCHEMA ===")
spark.table("workspace.gridpulse_bronze.consumption_bronze").printSchema()

=== BUILDINGS BRONZE SCHEMA ===
root
 |-- active_flag: boolean (nullable = true)
 |-- building_id: string (nullable = true)
 |-- building_name: string (nullable = true)
 |-- building_type: string (nullable = true)
 |-- campus_zone: string (nullable = true)
 |-- commissioning_date: string (nullable = true)
 |-- criticality_band: string (nullable = true)
 |-- floor_area_sqm: double (nullable = true)
 |-- operating_profile: string (nullable = true)
 |-- source_record_id: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)


=== CONSUMPTION BRONZE SCHEMA ===
root
 |-- source_record_id: string (nullable = true)
 |-- reading_id: string (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- reading_ts: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- active_power_kw: double (nullable = true)
 |-- voltage_v: double (nullable = true)
 |-- current_a: double (nullable = true)
 |-- power_facto

In [0]:
# Create Silver schema

spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.gridpulse_silver
""")

print("Silver schema ready")

Silver schema ready


In [0]:
# Create Silver meters table
# Standardize column names and keep appropriate data types

meters_silver = (
    spark.table("workspace.gridpulse_bronze.meters_bronze")
    .select(
        "meter_id",
        "meter_serial_no",
        "building_id",
        "tariff_plan_id",
        "meter_type",
        "capacity_kw",
        "meter_status",
        "voltage_class",
        "installed_date",
        "effective_from",
        "effective_to"
    )
)

meters_silver.write.mode("overwrite").saveAsTable(
    "workspace.gridpulse_silver.meters_silver"
)

print("Silver meters table created")
print("Row count:", meters_silver.count())

Silver meters table created
Row count: 120


In [0]:
# Create Silver buildings table
# Standardize commissioning_date from string to date

from pyspark.sql.functions import to_date

buildings_silver = (
    spark.table("workspace.gridpulse_bronze.buildings_bronze")
    .select(
        "building_id",
        "building_name",
        "building_type",
        "campus_zone",
        to_date("commissioning_date").alias("commissioning_date"),
        "criticality_band",
        "floor_area_sqm",
        "operating_profile",
        "active_flag"
    )
)

buildings_silver.write.mode("overwrite").saveAsTable(
    "workspace.gridpulse_silver.buildings_silver"
)

print("Silver buildings table created")
print("Row count:", buildings_silver.count())

Silver buildings table created
Row count: 5


In [0]:
# Create Silver consumption table
# Standardize the consumption fields and keep appropriate data types

consumption_silver = (
    spark.table("workspace.gridpulse_bronze.consumption_bronze")
    .select(
        "reading_id",
        "meter_id",
        "reading_ts",
        "energy_kwh",
        "active_power_kw",
        "voltage_v",
        "current_a",
        "power_factor",
        "reading_quality_flag",
        "source_system",
        "producer_run_id"
    )
)

consumption_silver.write.mode("overwrite").saveAsTable(
    "workspace.gridpulse_silver.consumption_silver"
)

print("Silver consumption table created")
print("Row count:", consumption_silver.count())

Silver consumption table created
Row count: 500


In [0]:
# Join Silver meters with Silver buildings
# Purpose: enrich meter records with building information

meter_building_silver = (
    spark.table("workspace.gridpulse_silver.meters_silver")
    .join(
        spark.table("workspace.gridpulse_silver.buildings_silver"),
        on="building_id",
        how="left"
    )
    .select(
        "meter_id",
        "meter_serial_no",
        "building_id",
        "building_name",
        "building_type",
        "campus_zone",
        "meter_type",
        "capacity_kw",
        "meter_status",
        "voltage_class",
        "tariff_plan_id"
    )
)

print("Meter-building join completed")
print("Row count:", meter_building_silver.count())

display(meter_building_silver)

Meter-building join completed
Row count: 120


meter_id,meter_serial_no,building_id,building_name,building_type,campus_zone,meter_type,capacity_kw,meter_status,voltage_class,tariff_plan_id
MTR0001,GP-100001,BLD001,GridPulse Academic Block 01,Academic,North,MAIN,100.0,ACTIVE,415V_3PH,PLAN-A
MTR0002,GP-100002,BLD001,GridPulse Academic Block 01,Academic,North,HVAC,150.0,ACTIVE,415V_3PH,PLAN-B
MTR0003,GP-100003,BLD001,GridPulse Academic Block 01,Academic,North,LIGHTING,90.0,ACTIVE,415V_3PH,PLAN-A
MTR0004,GP-100004,BLD001,GridPulse Academic Block 01,Academic,North,SUBMETER,130.0,ACTIVE,415V_3PH,PLAN-B
MTR0005,GP-100005,BLD001,GridPulse Academic Block 01,Academic,North,SUBMETER,130.0,ACTIVE,415V_3PH,PLAN-A
MTR0006,GP-100006,BLD001,GridPulse Academic Block 01,Academic,North,SUBMETER,120.0,ACTIVE,415V_3PH,PLAN-B
MTR0007,GP-100007,BLD002,GridPulse Laboratory Block 02,Laboratory,South,MAIN,150.0,ACTIVE,415V_3PH,PLAN-A
MTR0008,GP-100008,BLD002,GridPulse Laboratory Block 02,Laboratory,South,HVAC,130.0,ACTIVE,415V_3PH,PLAN-B
MTR0009,GP-100009,BLD002,GridPulse Laboratory Block 02,Laboratory,South,LIGHTING,130.0,ACTIVE,415V_3PH,PLAN-A
MTR0010,GP-100010,BLD002,GridPulse Laboratory Block 02,Laboratory,South,LAB_EQUIPMENT,150.0,ACTIVE,415V_3PH,PLAN-B


In [0]:
# Save enriched meter-building Silver table

meter_building_silver.write.mode("overwrite").saveAsTable(
    "workspace.gridpulse_silver.meter_building_silver"
)

print("Meter-building Silver table created")

Meter-building Silver table created


In [0]:
print("=== WEEK 05 SILVER TABLE VERIFICATION ===")

tables = [
    "meters_silver",
    "buildings_silver",
    "consumption_silver",
    "meter_building_silver"
]

for table in tables:
    df = spark.table(f"workspace.gridpulse_silver.{table}")
    print(f"{table:25} | {df.count()} rows")

=== WEEK 05 SILVER TABLE VERIFICATION ===
meters_silver             | 120 rows
buildings_silver          | 5 rows
consumption_silver        | 500 rows
meter_building_silver     | 120 rows


In [0]:
# Week 05 - Compact Silver Schema Evidence

for table in [
    "meters_silver",
    "buildings_silver",
    "consumption_silver",
    "meter_building_silver"
]:
    print(f"\n=== {table} ===")
    df = spark.table(f"workspace.gridpulse_silver.{table}")
    print("Columns:", df.columns)
    print("Data types:", df.dtypes)


=== meters_silver ===
Columns: ['meter_id', 'meter_serial_no', 'building_id', 'tariff_plan_id', 'meter_type', 'capacity_kw', 'meter_status', 'voltage_class', 'installed_date', 'effective_from', 'effective_to']
Data types: [('meter_id', 'string'), ('meter_serial_no', 'string'), ('building_id', 'string'), ('tariff_plan_id', 'string'), ('meter_type', 'string'), ('capacity_kw', 'double'), ('meter_status', 'string'), ('voltage_class', 'string'), ('installed_date', 'date'), ('effective_from', 'timestamp'), ('effective_to', 'timestamp')]

=== buildings_silver ===
Columns: ['building_id', 'building_name', 'building_type', 'campus_zone', 'commissioning_date', 'criticality_band', 'floor_area_sqm', 'operating_profile', 'active_flag']
Data types: [('building_id', 'string'), ('building_name', 'string'), ('building_type', 'string'), ('campus_zone', 'string'), ('commissioning_date', 'date'), ('criticality_band', 'string'), ('floor_area_sqm', 'double'), ('operating_profile', 'string'), ('active_flag'

In [0]:
# Week 05 - Raw to Silver Mapping Evidence

mapping = [
    ("Bronze: commissioning_date", "Silver: commissioning_date", "String → Date"),
    ("Bronze: meter_id", "Silver: meter_id", "String → String"),
    ("Bronze: building_id", "Silver: building_id", "String → String"),
    ("Bronze: energy_kwh", "Silver: energy_kwh", "Double → Double"),
    ("Bronze: reading_ts", "Silver: reading_ts", "Timestamp → Timestamp")
]

mapping_df = spark.createDataFrame(
    mapping,
    ["Bronze Field", "Silver Field", "Transformation"]
)

display(mapping_df)

Bronze Field,Silver Field,Transformation
Bronze: commissioning_date,Silver: commissioning_date,String → Date
Bronze: meter_id,Silver: meter_id,String → String
Bronze: building_id,Silver: building_id,String → String
Bronze: energy_kwh,Silver: energy_kwh,Double → Double
Bronze: reading_ts,Silver: reading_ts,Timestamp → Timestamp
